In [ ]:
%pip install torch transformers datasets pandas numpy
%pip install datasets transformers torch

In [ ]:
%pip install ntlk

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch

In [5]:
import nltk
nltk.download('punkt')  # Needed for word_tokenize
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\youst\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


In [14]:
dataset = load_dataset("rajpurkar/squad")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [20]:
train_dataset = dataset['train'].select(range(5000))
validation_dataset = dataset['validation'].select(range(2000))

In [21]:
from math import floor

# First 5 entries
print("First 5 entries:")
for i in range(5):
    print(train_dataset[i])
    print("-" * 80)

# Middle 5 entries
print("Middle 5 entries:")
middle_start = floor(len(train_dataset) / 2) - 2
for i in range(middle_start, middle_start + 5):
    print(train_dataset[i])
    print("-" * 80)

# Last 5 entries
print("Last 5 entries:")
for i in range(len(train_dataset) - 5, len(train_dataset)):
    print(train_dataset[i])
    print("-" * 80)


First 5 entries:
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}
-------------------------------------

In [26]:
train_df = pd.DataFrame(train_dataset) 

train_df['input'] = "Context: " + train_df['context'] + " Question: " + train_df['question']

# Extract answers
answers = [ans['text'][0] for ans in train_df['answers']]
answer_starts = [ans['answer_start'][0] for ans in train_df['answers']]

# Create a new DataFrame with the formatted input-output pairs
qa_df = pd.DataFrame({
    'input': train_df['input'],
    'answer': answers,
    'answer_start': answer_starts
})

qa_df['answer_end'] = qa_df['answer_start'] + qa_df['answer'].apply(lambda x: len(x.split()) - 1)

print(qa_df.head())


                                               input  \
0  Context: Architecturally, the school has a Cat...   
1  Context: Architecturally, the school has a Cat...   
2  Context: Architecturally, the school has a Cat...   
3  Context: Architecturally, the school has a Cat...   
4  Context: Architecturally, the school has a Cat...   

                                    answer  answer_start  answer_end  
0               Saint Bernadette Soubirous           515         517  
1                a copper statue of Christ           188         192  
2                        the Main Building           279         281  
3  a Marian place of prayer and reflection           381         387  
4       a golden statue of the Virgin Mary            92          98  


In [ ]:
val_df = pd.DataFrame(validation_dataset) 

val_df['input'] = "Context: " + val_df['context'] + " Question: " + val_df['question']

answers_val = [ans['text'][0] for ans in val_df['answers']]  

qa_val_df = pd.DataFrame({
    'input': val_df['input'],
    'answer': answers_val
})

In [24]:
import numpy as np
from nltk.tokenize import word_tokenize
import torch
from collections import Counter

In [27]:
# Tokenize each question and answer into word tokens
qa_df['question_tokens'] = qa_df['input'].apply(lambda x: word_tokenize(x.lower()))
qa_df['answer_tokens'] = qa_df['answer'].apply(lambda x: word_tokenize(x.lower()))

In [28]:
def load_glove_embeddings(file_path):
    embeddings_index = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    return embeddings_index

glove_file_path = 'glove.6B.100d.txt'
glove_embeddings = load_glove_embeddings(glove_file_path)
print(f"Loaded {len(glove_embeddings)} word vectors.")

Loaded 400000 word vectors.


In [29]:
# Flatten all tokens to build vocab
all_tokens = [token for tokens in qa_df['question_tokens'] for token in tokens] + \
             [token for tokens in qa_df['answer_tokens'] for token in tokens]

# Create vocab
vocab = ['<PAD>', '<UNK>'] + [word for word, _ in Counter(all_tokens).most_common()]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

In [30]:
embedding_dim = 100
embedding_matrix = np.zeros((len(word2idx), embedding_dim))

for word, index in word2idx.items():
    if word in glove_embeddings:
        embedding_matrix[index] = glove_embeddings[word]
    else:
        embedding_matrix[index] = np.random.normal(scale=0.6, size=(100,))

print(f"Embedding matrix shape: {embedding_matrix.shape}")

Embedding matrix shape: (14899, 100)


In [31]:
def tokens_to_ids(tokens, word2idx, max_length):
    ids = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]
    if len(ids) < max_length:
        ids += [word2idx['<PAD>']] * (max_length - len(ids))
    else:
        ids = ids[:max_length]
    return ids

max_length = 32

qa_df['question_ids'] = qa_df['question_tokens'].apply(lambda x: tokens_to_ids(x, word2idx, max_length))
qa_df['answer_ids'] = qa_df['answer_tokens'].apply(lambda x: tokens_to_ids(x, word2idx, max_length))

In [32]:
questions_tensor = torch.tensor(qa_df['question_ids'].tolist())
answers_tensor = torch.tensor(qa_df['answer_ids'].tolist())

In [33]:
# Now you have the answer start and end positions in the qa_df
answer_start_tensor = torch.tensor(qa_df['answer_start'].tolist())
answer_end_tensor = torch.tensor(qa_df['answer_end'].tolist())

In [35]:
from torch.utils.data import DataLoader, TensorDataset

# Use the input tensors (questions_tensor) and answer start/end positions as labels
dataset = TensorDataset(questions_tensor, answer_start_tensor, answer_end_tensor)

# Create a DataLoader for batching
batch_size = 8
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Check the first batch to ensure everything is correct
for batch in data_loader:
    print(batch)
    break

[tensor([[   22,    10,  1999,  7253,     3,  1058,     5,   265,     6,  1046,
             5,     2,   265,   360,   531,   151,     2,    93, 11541,   342,
           186,     7,   697,     3,   240,    17,     2,   115,   172,  3068,
            28,   364],
        [   22,    10,   704,   124,   112,  1181,  2229,   212,     7,     2,
            37,     3,    98,    14,     2,  2478,  4898,     3,    34,   232,
             2,  1068,   738,  4339,     7,     2,    81,    70,     4,     2,
            37,    12],
        [   22,    10, 11474, 11475,    70,    17,     2,   115,  2811,     2,
         12587,     5,  1315,    74,   123,   192,  7828,  2369,     5,    18,
           911, 12588,    19,     8,     2,   115,   172,     6,    23,  2481,
          5865,   769],
        [   22,    10,   126,  1477,   125,  1078,    13,  4914,     3,  2596,
             6,  8954,     4,    80,   134,    57,  5145,    88,   133,  2262,
          8955,    10,   730,  1477,     3,  3776,  1477, 

In [71]:
import torch
import torch.nn as nn
import torch.optim as optim

# class QAModel(nn.Module):
#     def __init__(self, vocab_size, embedding_matrix, hidden_size, num_layers=2):
#         super().__init__()

#         self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)

#         self.lstm = nn.LSTM(100, hidden_size, num_layers=num_layers, 
#                             batch_first=True, bidirectional=True)

#         self.fc_start = nn.Linear(hidden_size * 2, 1)  # For predicting the start position
#         self.fc_end = nn.Linear(hidden_size * 2, 1)    # For predicting the end position

#     def forward(self, x):
#         x = self.embedding(x)  # Embed input token IDs into vectors
#         lstm_out, _ = self.lstm(x)  # Pass through LSTM layers

#         start_logits = self.fc_start(lstm_out)  # Start position logits
#         end_logits = self.fc_end(lstm_out)      # End position logits

#         return start_logits, end_logits

class QAModel(nn.Module):
    def __init__(self, vocab_size, embedding_matrix, hidden_size, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)

        self.lstm = nn.LSTM(100, hidden_size, num_layers=num_layers, 
                            batch_first=True, bidirectional=True)

        self.fc_start = nn.Linear(hidden_size * 2, 1)  # For predicting the start position
        self.fc_end = nn.Linear(hidden_size * 2, 1)    # For predicting the end position

    def forward(self, input_ids):
        # Forward pass through the embedding layer
        x = self.embedding(input_ids)  # Embed input token IDs into vectors
        
        # Pass through the LSTM
        lstm_out, _ = self.lstm(x)  # LSTM output
        
        # Predict the start and end positions
        start_logits = self.fc_start(lstm_out)  # Start position logits
        end_logits = self.fc_end(lstm_out)      # End position logits

        return start_logits, end_logits



In [72]:
# Hyperparameters
vocab_size = len(word2idx)  # Size of the tokenizer's vocabulary
hidden_size = 256  # Hidden layer size
num_layers = 2     # Number of LSTM layers (stacked)

# Initialize the model
model = QAModel(vocab_size=len(word2idx), 
                embedding_matrix=embedding_matrix, 
                hidden_size=hidden_size, 
                num_layers=num_layers)

In [73]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Loss function and optimizer
loss_fn_start = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])  # Loss for the start position
loss_fn_end = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])    # Loss for the end position
optimizer = optim.Adam(model.parameters(), lr=5e-5)

In [78]:
import torch
import torch.nn as nn
import torch.optim as optim

def train_model(model, data_loader, optimizer, loss_fn_start, loss_fn_end, device, epochs=5):
    model.train()  # Set model to training mode

    for epoch in range(epochs):
        total_loss = 0
        total_start_loss = 0
        total_end_loss = 0
        total_correct_start = 0
        total_correct_end = 0

        for batch in data_loader:
            questions_batch, answer_start_batch, answer_end_batch = batch
            questions_batch = questions_batch.to(device)
            answer_start_batch = answer_start_batch.to(device)
            answer_end_batch = answer_end_batch.to(device)

            optimizer.zero_grad()

            # Forward pass
            start_logits, end_logits = model(questions_batch)

            # Compute loss for start and end positions
            start_loss = loss_fn_start(start_logits.squeeze(-1), answer_start_batch)
            end_loss = loss_fn_end(end_logits.squeeze(-1), answer_end_batch)

            # Total loss (start_loss + end_loss)
            loss = start_loss + end_loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update the model's parameters

            # Accumulate losses for averaging
            total_loss += loss.item()
            total_start_loss += start_loss.item()
            total_end_loss += end_loss.item()

            # Calculate accuracy for start and end predictions
            start_preds = torch.argmax(start_logits, dim=1)
            end_preds = torch.argmax(end_logits, dim=1)

            total_correct_start += (start_preds == answer_start_batch).sum().item()
            total_correct_end += (end_preds == answer_end_batch).sum().item()

        # Average loss and accuracy per epoch
        avg_loss = total_loss / len(data_loader)
        avg_start_loss = total_start_loss / len(data_loader)
        avg_end_loss = total_end_loss / len(data_loader)
        start_accuracy = total_correct_start / len(data_loader.dataset)
        end_accuracy = total_correct_end / len(data_loader.dataset)

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Start Loss: {avg_start_loss:.4f}, End Loss: {avg_end_loss:.4f}")
        print(f"Start Accuracy: {start_accuracy*100:.2f}%, End Accuracy: {end_accuracy*100:.2f}%")

    print("Training finished!")



In [79]:
# Example usage:
train_model(model, data_loader, optimizer, loss_fn_start, loss_fn_end, device, epochs=5)


IndexError: Target 110 is out of bounds.

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim

# class ImprovedQuestionAnsweringModel(nn.Module):
#     def __init__(self, vocab_size, embed_size, hidden_size, num_layers=2):  # Fixed typo in method name
#         super(ImprovedQuestionAnsweringModel, self).__init__()

#         self.embedding = nn.Embedding(vocab_size, embed_size)

#         self.lstm = nn.LSTM(embed_size, hidden_size, num_layers=num_layers, 
#                             batch_first=True, bidirectional=True)

#         self.fc = nn.Linear(hidden_size * 2, vocab_size)  # *2 because of bidirectionality

#     def forward(self, x):
#         x = self.embedding(x)  # Embed input token IDs into vectors
#         lstm_out, _ = self.lstm(x)  # Pass through LSTM layers
#         out = self.fc(lstm_out)  # Apply the fully connected layer
#         return out


# vocab_size = len(tokenizer.vocab)  # Size of the tokenizer's vocabulary
# embed_size = 128  # Size of the word embeddings
# hidden_size = 128  # Hidden layer size (this can be adjusted for better performance)
# num_layers = 2  # Number of LSTM layers (stacked)


# model = ImprovedQuestionAnsweringModel(vocab_size, embed_size, hidden_size, num_layers)


# device = torch.device("cpu")
# model.to(device)

# loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)  # Ignore padding tokens
# optimizer = optim.Adam(model.parameters(), lr=0.001)



import torch
import torch.nn as nn
import torch.optim as optim


class ImprovedQuestionAnsweringModelWithGloVe(nn.Module):
    def __init__(self, vocab_size, embedding_matrix, hidden_size, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)


        self.lstm = nn.LSTM(100, hidden_size, num_layers=num_layers, 
                            batch_first=True, bidirectional=True)

        self.fc = nn.Linear(hidden_size * 2, vocab_size)  # *2 because of bidirectionality

    def forward(self, x):
        x = self.embedding(x)  # Embed input token IDs into vectors
        lstm_out,  = self.lstm(x)  # Pass through LSTM layers
        out = self.fc(lstm_out)  # Apply the fully connected layer
        return out


Hyperparameters
vocab_size = len(word2idx)  # Size of the tokenizer's vocabulary
hidden_size = 256  # Hidden layer size
num_layers = 2     # Number of LSTM layers (stacked)


model = ImprovedQuestionAnsweringModelWithGloVe(vocab_size=len(word2idx), 
                                                embedding_matrix=embedding_matrix, 
                                                hidden_size=hidden_size, 
                                                num_layers=num_layers)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)  # Ignore padding tokens
optimizer = optim.Adam(model.parameters(), lr=5e-5)

In [17]:
# # Training loop
# def train_model(model, data_loader, optimizer, loss_fn, epochs=5):
#     model.train()  # Set model to training mode
#     for epoch in range(epochs):
#         total_loss = 0
#         for questions_batch, answers_batch in data_loader:
#             optimizer.zero_grad()

#             # Forward pass
#             output = model(questions_batch)
#             output = output.view(-1, vocab_size)  # Flatten the output to (batch_size * seq_len, vocab_size)
#             answers_batch = answers_batch.view(-1)  # Flatten answers to match the output shape

#             # Calculate the loss
#             loss = loss_fn(output, answers_batch)
#             loss.backward()  # Backpropagation
#             optimizer.step()  # Update the weights

#             total_loss += loss.item()

#         print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss / len(data_loader)}")

# # Train the model
# train_model(model, data_loader, optimizer, loss_fn)


def train_model(model, data_loader, optimizer, loss_fn, epochs=5):
    model.train()  # Set model to training mode
    for epoch in range(epochs):
        total_loss = 0
        for questions_batch, answers_batch in data_loader:
            questions_batch = questions_batch.to(device)
            answers_batch = answers_batch.to(device)

            optimizer.zero_grad()

            # Forward pass
            output = model(questions_batch)
            output = output.view(-1, vocab_size)  # Flatten the output to (batch_size * seq_len, vocab_size)
            answers_batch = answers_batch.view(-1)  # Flatten answers to match the output shape

            # Calculate the loss
            loss = loss_fn(output, answers_batch)
            loss.backward()  # Backpropagation
            optimizer.step()  # Update the weights

            total_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss / len(data_loader)}")


train_model(model, data_loader, optimizer, loss_fn)

Epoch [1/5], Loss: 5.592956181312805


KeyboardInterrupt: 

In [14]:
def evaluate_model(model, data_loader):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0

    with torch.no_grad():  # No need to track gradients during evaluation
        for questions_batch, answers_batch in data_loader:
            # Forward pass
            output = model(questions_batch)
            output = output.argmax(dim=2)  # Get the predicted tokens with the highest probability
            output = output.view(-1).cpu().numpy()  # Flatten and move to CPU for comparison

            answers_batch = answers_batch.view(-1).cpu().numpy()  # Flatten answers for comparison

            correct += (output == answers_batch).sum()
            total += len(answers_batch)

    accuracy = correct / total
    print(f"Accuracy: {accuracy * 100:.2f}%")

# Evaluate the model
evaluate_model(model, data_loader)


Accuracy: 6.54%


In [ ]:
def evaluate_model(model, data_loader, loss_fn, tokenizer, device):
    model.eval()  # Set the model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0
    exact_match = 0  # Counter for exact match

    with torch.no_grad():  # No need to track gradients during evaluation
        for questions_batch, answers_batch in data_loader:
            # Move data to GPU if available
            questions_batch = questions_batch.to(device)
            answers_batch = answers_batch.to(device)

            # Forward pass
            output = model(questions_batch)
            output = output.view(-1, len(word2idx))  # Flatten to (batch_size * seq_len, vocab_size)
            answers_batch = answers_batch.view(-1)  # Flatten answers to match the output shape
            loss = loss_fn(output, answers_batch)
            total_loss += loss.item()

            # Get predicted tokens (argmax along the vocab dimension)
            predicted_answer = output.argmax(dim=1).cpu().numpy()
            answers_batch = answers_batch.cpu().numpy()

            # Calculate exact match (EM) and total correct predictions
            correct += (predicted_answer == answers_batch).sum()
            total += len(answers_batch)

            # Exact match calculation
            for pred, true in zip(predicted_answer, answers_batch):
                if pred == true:
                    exact_match += 1

    # Calculate final metrics
    accuracy = correct / total
    em_score = exact_match / total  # Exact Match ratio
    avg_loss = total_loss / len(data_loader)

    print(f"Evaluation Metrics:")
    print(f"  - Loss: {avg_loss:.4f}")
    print(f"  - Accuracy: {accuracy * 100:.2f}%")
    print(f"  - Exact Match (EM): {em_score * 100:.2f}%")

    return accuracy, em_score, avg_loss

In [13]:
device = torch.device("cpu")
model.to(device)


val_data_loader = DataLoader(
    TensorDataset(torch.tensor(validation_dataset['question_tokens'].tolist()), 
                  torch.tensor(validation_dataset['answer_tokens'].tolist())),
    batch_size=8, shuffle=False)


evaluate_model(model, val_data_loader, loss_fn, tokenizer, device)

KeyError: "Column question_tokens not in the dataset. Current columns in the dataset: ['id', 'title', 'context', 'question', 'answers']"

In [ ]:
# EVALUATE WITH THIS

import torch

def evaluate_model(model, data_loader, loss_fn, device):
    model.eval()  # Set the model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0
    exact_match = 0

    with torch.no_grad():  # Disable gradient tracking for evaluation
        for questions_batch, answers_batch in data_loader:
            # Move to device (GPU or CPU)
            questions_batch = questions_batch.to(device)
            answers_batch = answers_batch.to(device)

            # Forward pass
            outputs = model(questions_batch)
            outputs = outputs.view(-1, outputs.shape[-1])  # (batch_size * seq_len, vocab_size)
            answers_batch = answers_batch.view(-1)         # Flatten targets

            # Compute loss
            loss = loss_fn(outputs, answers_batch)
            total_loss += loss.item()

            # Get predictions
            preds = outputs.argmax(dim=1)  # (batch_size * seq_len)
            correct += (preds == answers_batch).sum().item()
            total += answers_batch.size(0)

            # Exact match: every token in the sequence must be correct
            preds_seq = preds.view(questions_batch.size(0), -1).cpu()
            targets_seq = answers_batch.view(questions_batch.size(0), -1).cpu()
            exact_match += (preds_seq == targets_seq).all(dim=1).sum().item()

    accuracy = correct / total
    em_score = exact_match / len(data_loader.dataset)
    avg_loss = total_loss / len(data_loader)

    print(f"\nEvaluation Metrics:")
    print(f"  - Loss: {avg_loss:.4f}")
    print(f"  - Accuracy: {accuracy * 100:.2f}%")
    print(f"  - Exact Match (EM): {em_score * 100:.2f}%")

    return accuracy, em_score, avg_loss


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

val_dataset = TensorDataset(
    torch.tensor(validation_dataset['question_tokens'].tolist()),
    torch.tensor(validation_dataset['answer_tokens'].tolist())
)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Call the function
evaluate_model(model, val_loader, loss_fn, device)
